In [1]:
# Install (only once)
import yfinance as yf
import pandas as pd

# Download NIFTY 50 data from 2000
df = yf.download("^NSEI", start="2000-01-01", interval="1d")

# Keep only required columns (OHLCV)
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

# Drop missing values (important)
df = df.dropna()

# Reset index (optional but useful)
df.reset_index(inplace=True)

# Preview
print(df.head())
print(df.tail())

# Check size
print("Total rows:", len(df))

/tmp/ipykernel_2459/3861514355.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("^NSEI", start="2000-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed

Price        Date         Open         High          Low        Close Volume
Ticker                   ^NSEI        ^NSEI        ^NSEI        ^NSEI  ^NSEI
0      2007-09-17  4518.450195  4549.049805  4482.850098  4494.649902      0
1      2007-09-18  4494.100098  4551.799805  4481.549805  4546.200195      0
2      2007-09-19  4550.250000  4739.000000  4550.250000  4732.350098      0
3      2007-09-20  4734.850098  4760.850098  4721.149902  4747.549805      0
4      2007-09-21  4752.950195  4855.700195  4733.700195  4837.549805      0
Price        Date          Open          High           Low         Close  \
Ticker                    ^NSEI         ^NSEI         ^NSEI         ^NSEI   
4545   2026-03-30  22549.650391  22714.099609  22283.849609  22331.400391   
4546   2026-04-01  22899.000000  22941.300781  22618.599609  22679.400391   
4547   2026-04-02  22383.400391  22782.300781  22182.550781  22713.099609   
4548   2026-04-06  22780.300781  22998.349609  22542.949219  22968.250000   

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
LOOKAHEAD = 5
THRESHOLD = 0.01  # 1%

labels = []

close_prices = df['Close'].values

for i in range(len(df)):
    if i + LOOKAHEAD >= len(df):
        labels.append(None)
        continue

    current_price = close_prices[i]
    future_window = close_prices[i+1 : i+1+LOOKAHEAD]

    future_max = np.max(future_window)
    future_min = np.min(future_window)

    up_move = (future_max - current_price) / current_price
    down_move = (future_min - current_price) / current_price

    if up_move >= THRESHOLD:
        labels.append("BUY")
    elif down_move <= -THRESHOLD:
        labels.append("SELL")
    else:
        labels.append("HOLD")

df['Label'] = labels
df = df.dropna()

In [4]:
label_map = {"BUY": 0, "SELL": 1, "HOLD": 2}
df['LabelEncoded'] = df['Label'].map(label_map)

In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Open','High','Low','Close','Volume']])

In [10]:
SEQUENCE_LENGTH = 60

X = []
y = []

for i in range(SEQUENCE_LENGTH, len(scaled_data)):
    X.append(scaled_data[i-SEQUENCE_LENGTH:i])
    y.append(df['LabelEncoded'].iloc[i])

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

(4485, 60, 5) (4485,)


In [11]:
split = int(0.8 * len(X))

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

In [14]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(60,5)),
    LSTM(32),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 60, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,491 (123.01 KB)

 Trainable params: 31,491 (123.01 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.5053 - loss: 0.9945 - val_accuracy: 0.4359 - val_loss: 1.1096
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.5047 - loss: 0.9813 - val_accuracy: 0.4359 - val_loss: 1.0903
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.5117 - loss: 0.9762 - val_accuracy: 0.4359 - val_loss: 1.0827
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 9s 53ms/step - accuracy: 0.5164 - loss: 0.9785 - val_accuracy: 0.4359 - val_loss: 1.0914
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.5148 - loss: 0.9710 - val_accuracy: 0.4359 - val_loss: 1.1205
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - accuracy: 0.5176 - loss: 0.9666 - val_accuracy: 0.4359 - val_loss: 1.1277
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.5181 - loss: 0.9659 - val_accuracy: 0.4359 - val_loss: 1.1028
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - accuracy: 0.5184 - loss: 0.9601 - val_a

In [16]:
model.save("lstm_signal_model.h5")

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
model.save("/content/drive/MyDrive/lstm_signal_model.h5")

In [19]:
import joblib

joblib.dump(scaler, "/content/drive/MyDrive/scaler.save")

['/content/drive/MyDrive/scaler.save']